In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:
####CONSULTA DE ARCHIVOS #####
from google.cloud import storage
import zipfile
import io
import re
import os

# Configuración
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
PREFIX = "data_entries/Tramas Desgravamen/"  # Carpeta en el bucket
# PATTERN = re.compile(r"RED\d{6}_\d{4}\.txt$", re.IGNORECASE)

# Expresiónregular para el patrón
PATTERN = re.compile(r"RED(\d{4}_\d{6}_?|\d{6}_\d{4}_?)\.txt$", re.IGNORECASE)

def list_txt_files_with_pattern(bucket_name, prefix, pattern):
    """Lista los nombres de archivos TXT dentro de archivos ZIP, incluyendo aquellos en subcarpetas."""
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blobs = bucket.list_blobs(prefix=prefix)

    # Iterar sobre todos los archivos en la ruta especificada
    for blob in blobs:
        if blob.name.endswith('.zip'):  # Filtrar solo archivos ZIP
            print(f"Procesando archivo ZIP: {blob.name}")

            # Descargar el ZIP en memoria
            zip_bytes = blob.download_as_bytes()

            # Abrir el ZIP en memoria
            with zipfile.ZipFile(io.BytesIO(zip_bytes), 'r') as z:
                for file_path in z.namelist():  # Recorremos todos los archivos dentro del ZIP
                    file_name = os.path.basename(file_path)  # Extraer solo el nombre del archivo ignorando carpetas
                    if pattern.match(file_name):  # Aplicar la expresión regular al nombre limpio
                        print(f"Archivo encontrado: {file_name} en {blob.name} (Ruta interna: {file_path})")

# Ejecutar la función
list_txt_files_with_pattern(BUCKET_NAME, PREFIX, PATTERN)


Procesando archivo ZIP: data_entries/Tramas Desgravamen/04_Abril25.zip
Archivo encontrado: RED010425_0204.TXT en data_entries/Tramas Desgravamen/04_Abril25.zip (Ruta interna: 04_Abril25/RED010425_0204.TXT)
Archivo encontrado: RED020425_0304.TXT en data_entries/Tramas Desgravamen/04_Abril25.zip (Ruta interna: 04_Abril25/RED020425_0304.TXT)
Archivo encontrado: RED030425_0404.TXT en data_entries/Tramas Desgravamen/04_Abril25.zip (Ruta interna: 04_Abril25/RED030425_0404.TXT)
Archivo encontrado: RED040425_0704.TXT en data_entries/Tramas Desgravamen/04_Abril25.zip (Ruta interna: 04_Abril25/RED040425_0704.TXT)
Archivo encontrado: RED070425_0804.TXT en data_entries/Tramas Desgravamen/04_Abril25.zip (Ruta interna: 04_Abril25/RED070425_0804.TXT)
Archivo encontrado: RED080425_0904.TXT en data_entries/Tramas Desgravamen/04_Abril25.zip (Ruta interna: 04_Abril25/RED080425_0904.TXT)
Archivo encontrado: RED090425_1004.TXT en data_entries/Tramas Desgravamen/04_Abril25.zip (Ruta interna: 04_Abril25/RED0

# versión anterior:

In [ ]:
"""
Código COMPLETO E INTEGRADO con verificación de dataset y tabla de control.
"""

import zipfile
import io
import os
import re
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from datetime import datetime
from google.cloud import storage, bigquery
from google.cloud.exceptions import NotFound
from itertools import accumulate


# =====================
# CONFIGURACIÓN
# =====================
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dt-ue4-gcs-royspc-sftp_custom"
PREFIX = "TRAMAS_DIARIAS/2025/mes_prueba/"
DATASET_ID = "develop"
TABLE_PRIMA_ID = "TABLA_PRESTAMOS_BBVA"
TABLE_CONTROL_ID = "TABLA_CONTROL_PRESTAMOS"
GCS_TEMP_PATH = f"gs://{BUCKET_NAME}/temp_parquet_ptmos/"

# =====================
# CLIENTES DE GCP
# =====================
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

# =====================
# EXPRESIÓN REGULAR
# =====================
PATTERN = re.compile(r"^SUST(\d{2})(\d{2})(?:_\d+)?\.TXT$", re.IGNORECASE)

# =====================
# ESTRUCTURA DEL TXT
# =====================

# Definir los anchos de columna
col_widths = [3, 20, 20, 2, 3, 1, 30, 30, 30, 1, 1, 8, 1, 10, 50, 25, 25, 30, 8, 8, 30, 2, 5, 3, 1, 3, 3, 2, 3, 3, 7, 8, 8, 8, 4, 15, 15, 1, 2, 1, 1, 1, 10, 7, 73, 2, 3, 3, 10, 10, 15, 70, 2, 2, 2]

column_names = ["Tipo de seguro", "Certificado", "Numero Interno Del Canal", "Tipo de Registro", "Moneda", "Tipo de Movimiento", "Apellido Paterno", "Apellido Materno", "Nombres", "Sexo", "Estado Civil", "Fecha de Nacimiento", "Tipo de Documento de Identidad", "Numero de Documento de Identidad", "Dirección Domiciliaria", "Referencia", "Referencia2", "Urbanización", "Mz/Lte/Nro", "Apto/Int", "Distrito", "Departamento", "Código Postal", "País", "Periodo de pago", "Prefijo Calle,Av,Jr", "Prefijo 2", "Provincia", "Distrito2", "Prefijo Teléfono", "Numero Teléfono", "Fecha de Afiliación", "Fecha de inicio del seguro", "Fecha fin del seguro", "Plazo del seguro", "Monto Asegurado", "Prima", "Período de Gracia", "Código de Beneficiario", "Clase de Prima", "Tipo de Titular", "Mayor a 65 Años", "Tasa sin Recargo", "% de recargo", "Filler", "Tiempo de Periodo de Gracia", "Tipos de Endoso", "Tipos de Extorno", "Tipo de cambio", "Póliza", "Sumatoria de Capital en Cúmulo", "Correo electrónico", "Plan de crédito", "Planes Tipo Desgravamen", "Planes de Seguro (o Modalidad)"]


# =====================
# LOG
# =====================
def log(message: str) -> None:
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")


# =====================
# EXPRESIÓN REGULAR -> FECHA
# =====================

def extract_fecha_recepcion(file_name: str) -> str:
    match = PATTERN.match(file_name)
    if not match:
        return None

    mm, aa = match.groups()

    # Convertir año
    yy = int(aa)
    full_year = 2000 + yy if yy < 50 else 1900 + yy  # Si yy < 50 => 20xx, de lo contrario 19xx

    # Día fijo = 15
    return f"{full_year}-{mm}-15"

# =====================
# TABLA PRINCIPAL
# =====================
def create_table_if_not_exists():
    """
    Crea la tabla principal en BigQuery con particionamiento
    si no existe. Si ya existe, no se modifica su esquema.
    En esta versión, el campo 'Prima' se define como FLOAT.
    """
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_PRIMA_ID}"

    # Definir el esquema:
    # - Para 'Prima' se define FLOAT.
    # - Para el resto de las columnas se utiliza STRING.
    schema = []
    for col in column_names:
        if col in ["Monto asegurado", "Prima", "Sumatoria de Capital en Cúmulo"]:
            schema.append(bigquery.SchemaField(col, "FLOAT"))
        else:
            schema.append(bigquery.SchemaField(col, "STRING"))
    schema += [
        bigquery.SchemaField("trama_original", "STRING"),
        bigquery.SchemaField("nombre_archivo_trama", "STRING"),
        bigquery.SchemaField("fecha_recepcion", "DATE"),
    ]

    # ==== Imprimir el esquema que se generó ====
    log(">>> Esquema a usar para crear la tabla:")
    # for field in schema:
    #     log(f" - {field.name}: {field.field_type}")

    try:
        bigquery_client.get_table(table_id)
        log(f"✅ La tabla {TABLE_PRIMA_ID} ya existe (no se modifica).")
    except NotFound:
        # Crear la tabla con particionamiento únicamente
        table = bigquery.Table(table_id, schema=schema)

        # Particionamiento por fecha_recepcion
        table.time_partitioning = bigquery.TimePartitioning(field="fecha_recepcion")

        bigquery_client.create_table(table)
        log(f"✅ Tabla {TABLE_PRIMA_ID} creada con particionamiento en BigQuery.")

# =====================
# TABLA DE CONTROL
# =====================
def create_control_table():
    """
    Crea la tabla de control en BigQuery si no existe.
    """
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_CONTROL_ID}"

    schema = [
        bigquery.SchemaField("nombre_archivo", "STRING"),
        bigquery.SchemaField("fecha_carga", "TIMESTAMP"),
        bigquery.SchemaField("estado", "STRING"),
    ]

    try:
        bigquery_client.get_table(table_id)
        log(f"✅ La tabla de control {TABLE_CONTROL_ID} ya existe.")
    except NotFound:
        table = bigquery.Table(table_id, schema=schema)
        bigquery_client.create_table(table)
        log(f"✅ Tabla de control {TABLE_CONTROL_ID} creada.")
        # Pequeño retardo para asegurar la propagación de metadatos
        time.sleep(5)

# =====================
# VALIDACIÓN DE PROCESADO DE ZIP
# =====================
def check_if_zip_processed(file_name: str) -> bool:
    """Verifica si el ZIP ya fue procesado (estado = 'CARGADO') en la tabla de control."""
    query = f"""
        SELECT COUNT(*) AS count
        FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_CONTROL_ID}`
        WHERE nombre_archivo = '{file_name}' AND estado = 'CARGADO'
    """
    df = bigquery_client.query(query).to_dataframe()
    return df['count'][0] > 0

# =====================
# VALIDACIÓN DE PROCESADO DE PARQUET
# =====================
def check_if_parquet_processed(parquet_name: str) -> bool:
    """Verifica si el Parquet ya fue procesado (estado = 'CARGADO_PARQUET') en la tabla de control."""
    query = f"""
        SELECT COUNT(*) AS count
        FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_CONTROL_ID}`
        WHERE nombre_archivo = '{parquet_name}' AND estado = 'CARGADO_PARQUET'
    """
    df = bigquery_client.query(query).to_dataframe()
    return df['count'][0] > 0

# =====================
# REGISTRO EN TABLA DE CONTROL CON REINTENTOS
# =====================
def insert_control_record(file_name: str, estado: str, max_retries: int = 3, delay: int = 5) -> None:
    """
    Registra el estado del archivo (ZIP o PARQUET) en la tabla de control.
    - 'CARGADO' para ZIP
    - 'CARGADO_PARQUET' para Parquet
    Se reintenta la inserción en caso de error NotFound.
    """
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_CONTROL_ID}"
    rows_to_insert = [{
        "nombre_archivo": file_name,
        "fecha_carga": datetime.utcnow().isoformat(),
        "estado": estado
    }]

    for attempt in range(1, max_retries + 1):
        try:
            errors = bigquery_client.insert_rows_json(table_id, rows_to_insert)
            if errors:
                log(f"❌ Errores al insertar en la tabla de control: {errors}")
            else:
                log(f"✅ Registro insertado en tabla de control para {file_name} (estado: {estado}).")
            break
        except NotFound:
            log(f"❌ Intento {attempt}: La tabla {TABLE_CONTROL_ID} no se encontró. Esperando {delay} segundos...")
            time.sleep(delay)
    else:
        log(f"❌ Error: No se pudo insertar el registro en {TABLE_CONTROL_ID} tras {max_retries} intentos.")

# =====================
# FUNCIÓN DE CONVERSIÓN ESPECÍFICA
# =====================
def convert_prima(series: pd.Series) -> pd.Series:
    """
    Optimiza la conversión para columnas en las que se reemplaza el último carácter
    según un diccionario, por ejemplo: 'Plazo_del_seguro', 'Monto_Asegurado', 'Prima'.
    """
    clave = np.array(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ{0123456789"))
    valor = np.array(list("1234567891234567890000000000123456789"))
    diccionario_reemplazo = dict(zip(clave, valor))

    series = series.fillna("0").astype(str)

    return pd.Series(
        np.where(
            (series == "0") | (series == ""),
            series,
            series.str[:-1] + series.str[-1].map(diccionario_reemplazo).fillna("")
        ),
        index=series.index
    )

# =====================
# PROCESAMIENTO DE TXT
# =====================
def process_txt_file(txt_file, file_name: str) -> pd.DataFrame:
    """
    Procesa un archivo TXT a DataFrame y lo guarda en Parquet en GCS.

    Parámetros:
      - txt_file: Archivo TXT abierto en modo binario.
      - file_name: Nombre del archivo (se utiliza para extraer la fecha y registrar en control).

    Retorna:
      - pd.DataFrame con los datos procesados.
    """
    fecha_recepcion = extract_fecha_recepcion(file_name)
    if not fecha_recepcion:
        log(f"❌ No se pudo extraer la fecha de recepción del archivo: {file_name}")
        return pd.DataFrame()

    # Leer y decodificar el contenido del archivo
    lines = txt_file.read().decode('latin-1').splitlines()

    # Precalcular los índices de corte para cada columna utilizando accumulate
    cumulative_indices = [0] + list(accumulate(col_widths))

    data = []
    for line in lines:
        # Extraer cada campo de acuerdo a los anchos definidos y quitar espacios en blanco
        row = {
            col: line[start:end].strip()
            for col, start, end in zip(column_names, cumulative_indices[:-1], cumulative_indices[1:])
        }
        # Agregar campos adicionales
        row.update({
            "trama_original": line.rstrip("\n"),
            "nombre_archivo_trama": file_name,
            "fecha_recepcion": fecha_recepcion
        })
        data.append(row)

    # Crear DataFrame con las columnas en el orden esperado
    df = pd.DataFrame(data, columns=column_names + ["trama_original", "nombre_archivo_trama", "fecha_recepcion"])

    # Convertir "fecha_recepcion" a tipo fecha
    df["fecha_recepcion"] = pd.to_datetime(df["fecha_recepcion"], format="%Y-%m-%d").dt.date

    # Lista de columnas que requieren conversión a numérico (dividido entre 100)
    columnas_primas = ["Monto asegurado", "Prima", "Sumatoria de Capital en Cúmulo"]
    for col in columnas_primas:
        if col in df.columns:
            df[col] = pd.to_numeric(convert_prima(df[col]), errors="coerce") / 100.0

    df["Plazo del seguro"] = pd.to_numeric(convert_prima(df["Plazo del seguro"]), errors="coerce")

    # Guardar el DataFrame en formato Parquet en GCS
    file_parquet = f"{GCS_TEMP_PATH}{file_name}.parquet"
    table = pa.Table.from_pandas(df)
    pq.write_table(table, file_parquet)
    log(f"📤 Guardado {file_parquet} en GCS.")

    return df

# =====================
# CARGA PARQUET A BQ (Control de duplicados)
# =====================
def load_parquet_to_bigquery():
    """
    Carga los archivos Parquet de GCS a la tabla principal en BigQuery,
    evitando duplicados mediante la tabla de control.
    """
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_PRIMA_ID}"
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition="WRITE_APPEND"
    )

    bucket = storage_client.bucket(BUCKET_NAME)
    prefix_parquet = GCS_TEMP_PATH.replace(f"gs://{BUCKET_NAME}/", "")
    blobs = list(bucket.list_blobs(prefix=prefix_parquet))

    if not blobs:
        log("⚠️ No se encontraron archivos PARQUET en la ruta.")
        return

    for blob in blobs:
        if blob.name.endswith(".parquet"):
            parquet_name = os.path.basename(blob.name)

            # Verificar si ya se cargó
            if check_if_parquet_processed(parquet_name):
                log(f"🔄 Parquet ya cargado: {parquet_name}. Omitiendo...")
                continue

            # Cargar este Parquet en BQ
            parquet_uri = f"gs://{BUCKET_NAME}/{blob.name}"
            log(f"🚀 Cargando {parquet_name} a BigQuery...")

            load_job = bigquery_client.load_table_from_uri(
                parquet_uri,
                table_id,
                job_config=job_config
            )
            load_job.result()

            log(f"✅ Cargado Parquet {parquet_name} en BigQuery.")
            insert_control_record(parquet_name, "CARGADO_PARQUET")

    log("🚀 Carga de todos los Parquets completada en BigQuery.")

# =====================
# PROCESAR ZIP
# =====================
def process_zip_files():
    """
    Procesa archivos ZIP en GCS, extrayendo los que matchean PATTERN,
    incluso en subcarpetas, y evita reprocesarlos si el Parquet ya existe.
    """
    bucket = storage_client.bucket(BUCKET_NAME)
    blobs = list(bucket.list_blobs(prefix=PREFIX))

    if not blobs:
        log("⚠️ No se encontraron archivos en el bucket.")
        return

    for blob in blobs:
        if blob.name.endswith(".zip"):
            file_name = os.path.basename(blob.name)

            # Chequear si el ZIP ya fue procesado
            if check_if_zip_processed(file_name):
                log(f"🔄 Archivo ZIP ya procesado: {file_name}. Omitiendo...")
                continue

            log(f"\n📦 Procesando ZIP: {file_name}")
            zip_bytes = blob.download_as_bytes()

            with zipfile.ZipFile(io.BytesIO(zip_bytes), "r") as z:
                all_files = z.namelist()
                log(f"Archivos dentro del ZIP: {all_files}")

                # Filtrar archivos .txt que matchean la regex
                archivos_txt = []
                for path_in_zip in all_files:
                    base = os.path.basename(path_in_zip)
                    if PATTERN.match(base):
                        archivos_txt.append(path_in_zip)

                if not archivos_txt:
                    log(f"⚠️ No se encontraron TXT válidos en {file_name}. Omitiendo...")
                    continue

                for txt_path in archivos_txt:
                    base_txt_name = os.path.basename(txt_path)
                    parquet_file = f"{base_txt_name}.parquet"
                    full_parquet_gcs_path = f"{GCS_TEMP_PATH}{parquet_file}"

                    # Revisamos si el Parquet ya existe en el bucket
                    parquet_blob = bucket.blob(full_parquet_gcs_path.replace(f"gs://{BUCKET_NAME}/", ""))
                    if parquet_blob.exists():
                        log(f"🔄 Parquet {parquet_file} ya existe. Omitiendo procesamiento de {base_txt_name}...")
                        continue

                    log(f"→ Procesando archivo: {txt_path} (basename={base_txt_name})")
                    with z.open(txt_path) as txt_file:
                        df = process_txt_file(txt_file, base_txt_name)
                        if not df.empty:
                            # Marcar el ZIP como procesado
                            insert_control_record(file_name, "CARGADO")

# =====================
# MAIN
# =====================
if __name__ == "__main__":
    log("🚀 Iniciando proceso...")

    # 2. Crear tablas si no existen, con particionamiento y clustering
    create_table_if_not_exists()
    create_control_table()

    # 3. Procesar ZIP y extraer a Parquet (evita recrear Parquet si ya existe)
    process_zip_files()

    # 4. Cargar los Parquets a la tabla principal (evitando duplicados)
    load_parquet_to_bigquery()

    log("✅ Proceso finalizado.")


[2025-02-06 22:21:51] 🚀 Iniciando proceso...
[2025-02-06 22:21:51] >>> Esquema a usar para crear la tabla:
[2025-02-06 22:21:52] ✅ Tabla TABLA_SUSTENTO_BBVA creada con particionamiento en BigQuery.
[2025-02-06 22:21:53] ✅ Tabla de control TABLA_CONTROL_SUSTENTO creada.
[2025-02-06 22:22:00] 
📦 Procesando ZIP: SUST1223.zip
[2025-02-06 22:22:01] Archivos dentro del ZIP: ['SUST1223.TXT']
[2025-02-06 22:22:01] → Procesando archivo: SUST1223.TXT (basename=SUST1223.TXT)
[2025-02-06 22:23:27] 📤 Guardado gs://rs-nprd-dt-ue4-gcs-royspc-sftp_custom/temp_parquet_sust/SUST1223.TXT.parquet en GCS.
[2025-02-06 22:23:29] ✅ Registro insertado en tabla de control para SUST1223.zip (estado: CARGADO).
[2025-02-06 22:23:34] 🚀 Cargando SUST1223.TXT.parquet a BigQuery...
[2025-02-06 22:23:54] ✅ Cargado Parquet SUST1223.TXT.parquet en BigQuery.
[2025-02-06 22:23:55] ✅ Registro insertado en tabla de control para SUST1223.TXT.parquet (estado: CARGADO_PARQUET).
[2025-02-06 22:23:55] 🚀 Carga de todos los Parquet

# MOSTRAR

In [3]:
"""
Código COMPLETO E INTEGRADO con verificación de dataset, tabla de control y filtrado para préstamos.
"""

import zipfile
import io
import os
import re
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from datetime import datetime
from google.cloud import storage, bigquery
from google.cloud.exceptions import NotFound
from itertools import accumulate

# =====================
# CONFIGURACIÓN
# =====================
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
PREFIX = "data_entries/Tramas Desgravamen/"
DATASET_ID = "develop"
TABLE_PRIMA_ID = "TABLA_PRESTAMOS_BBVA"
TABLE_CONTROL_ID = "TABLA_CONTROL_PREST"
GCS_TEMP_PATH = f"gs://{BUCKET_NAME}/temp_parquet_prest/"

# =====================
# CLIENTES DE GCP
# =====================
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

# =====================
# EXPRESIÓN REGULAR
# =====================
# Este patrón captura dos grupos: uno de 4 o 6 dígitos y otro de 4 o 6 dígitos.
PATTERN = re.compile(r"^RED(\d{4}|\d{6})_(\d{4}|\d{6})\.txt$", re.IGNORECASE)

# =====================
# ESTRUCTURA DEL TXT
# =====================
col_widths = [3, 20, 20, 2, 3, 1, 30, 30, 30, 1, 1, 8, 1, 10, 50, 25, 25, 30, 8, 8, 30, 2, 5, 3, 1, 3, 3, 2, 3, 3, 7, 8, 8, 8, 4, 15, 15, 1, 2, 1, 1, 1, 10, 7, 73, 2, 3, 3, 10, 10, 15, 70, 2, 2, 2]

column_names = ['Tipo_de_seguro', 'Certificado', 'Numero_Interno_Canal', 'Tipo_Registro', 'Moneda', 'Tipo_Movimiento', 'Apellido_Paterno', 'Apellido_Materno', 'Nombres', 'Sexo', 'Estado_Civil', 'Fecha_Nacimiento', 'Tipo_Documento_Identidad', 'Numero_Documento_Identidad', 'Direccion_Domiciliaria', 'Referencia', 'Referencia2', 'Urbanizacion', 'Mz_Lte_Nro', 'Apto_Int', 'Distrito1', 'Departamento', 'Codigo_Postal', 'País', 'Periodo_pago', 'Prefijo_Calle_Av_Jr', 'Prefijo', 'Provincia', 'Distrito', 'Prefijo_Telefono', 'Numero_Telefono', 'Fecha_Afiliacion', 'Fecha_Inicio_Seguro', 'Fecha_Fin_Seguro', 'Plazo_Seguro', 'Monto_Asegurado', 'Prima', 'Periodo_Gracia', 'Codigo_Beneficiario', 'Clase_Prima', 'Tipo_Titular', 'Mayor_65Años', 'Tasa_sin_Recargo', '%_recargo', 'Filler', 'Tiempo_Periodo_Gracia', 'Tipos_Endoso', 'Tipos_Extorno', 'Tipo_de_cambio', 'Poliza', 'Sumatoria_Capital_Cumulo', 'Correo_electronico', 'Plan_credito', 'Planes_Tipo_Desgravamen', 'Planes_Seguro_Modalidad']

# =====================
# LOG
# =====================
def log(message: str) -> None:
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")

# =====================
# EXPRESIÓN REGULAR -> FECHA
# =====================
def extract_fecha_recepcion(file_name: str) -> str:
    match = PATTERN.match(file_name)
    if not match:
        return None

    g1, g2 = match.groups()

    def parse_ddmmaa(s: str):
        dd = s[:2]
        mm = s[2:4]
        yy = int(s[4:])
        full_year = 2000 + yy if yy < 50 else 1900 + yy
        return full_year, mm, dd

    def parse_ddmm(s: str):
        dd = s[:2]
        mm = s[2:]
        return dd, mm

    if len(g1) == 6 and len(g2) == 4:
        year, _, _ = parse_ddmmaa(g1)
        dd, mm = parse_ddmm(g2)
    elif len(g1) == 4 and len(g2) == 6:
        dd, mm = parse_ddmm(g1)
        year, _, _ = parse_ddmmaa(g2)
    else:
        return None

    return f"{year}-{mm}-{dd}"

# =====================
# TABLA PRINCIPAL
# =====================
def create_table_if_not_exists():
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_PRIMA_ID}"
    schema = []
    for col in column_names:
        if col in ["Monto_asegurado", "Prima", "Sumatoria_Capital_Cumulo"]:
            schema.append(bigquery.SchemaField(col, "FLOAT"))
        else:
            schema.append(bigquery.SchemaField(col, "STRING"))
    schema += [
        bigquery.SchemaField("trama_original", "STRING"),
        bigquery.SchemaField("nombre_archivo_trama", "STRING"),
        bigquery.SchemaField("fecha_recepcion", "DATE"),
    ]
    log(">>> Esquema a usar para crear la tabla:")
    try:
        bigquery_client.get_table(table_id)
        log(f"✅ La tabla {TABLE_PRIMA_ID} ya existe (no se modifica).")
    except NotFound:
        table = bigquery.Table(table_id, schema=schema)
        table.time_partitioning = bigquery.TimePartitioning(field="fecha_recepcion")
        bigquery_client.create_table(table)
        log(f"✅ Tabla {TABLE_PRIMA_ID} creada con particionamiento en BigQuery.")

# =====================
# TABLA DE CONTROL
# =====================
def create_control_table():
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_CONTROL_ID}"
    schema = [
        bigquery.SchemaField("nombre_archivo", "STRING"),
        bigquery.SchemaField("fecha_carga", "TIMESTAMP"),
        bigquery.SchemaField("estado", "STRING"),
    ]
    try:
        bigquery_client.get_table(table_id)
        log(f"✅ La tabla de control {TABLE_CONTROL_ID} ya existe.")
    except NotFound:
        table = bigquery.Table(table_id, schema=schema)
        bigquery_client.create_table(table)
        log(f"✅ Tabla de control {TABLE_CONTROL_ID} creada.")
        time.sleep(5)

# =====================
# VALIDACIÓN DE PROCESADO DE ZIP
# =====================
def check_if_zip_processed(file_name: str) -> bool:
    query = f"""
        SELECT COUNT(*) AS count
        FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_CONTROL_ID}`
        WHERE nombre_archivo = '{file_name}' AND estado = 'CARGADO'
    """
    df = bigquery_client.query(query).to_dataframe()
    return df['count'][0] > 0

# =====================
# VALIDACIÓN DE PROCESADO DE PARQUET
# =====================
def check_if_parquet_processed(parquet_name: str) -> bool:
    query = f"""
        SELECT COUNT(*) AS count
        FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_CONTROL_ID}`
        WHERE nombre_archivo = '{parquet_name}' AND estado = 'CARGADO_PARQUET'
    """
    df = bigquery_client.query(query).to_dataframe()
    return df['count'][0] > 0

# =====================
# REGISTRO EN TABLA DE CONTROL CON REINTENTOS
# =====================
def insert_control_record(file_name: str, estado: str, max_retries: int = 3, delay: int = 5) -> None:
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_CONTROL_ID}"
    rows_to_insert = [{
        "nombre_archivo": file_name,
        "fecha_carga": datetime.utcnow().isoformat(),
        "estado": estado
    }]
    for attempt in range(1, max_retries + 1):
        try:
            errors = bigquery_client.insert_rows_json(table_id, rows_to_insert)
            if errors:
                log(f"❌ Errores al insertar en la tabla de control: {errors}")
            else:
                log(f"✅ Registro insertado en tabla de control para {file_name} (estado: {estado}).")
            break
        except NotFound:
            log(f"❌ Intento {attempt}: La tabla {TABLE_CONTROL_ID} no se encontró. Esperando {delay} segundos...")
            time.sleep(delay)
    else:
        log(f"❌ Error: No se pudo insertar el registro en {TABLE_CONTROL_ID} tras {max_retries} intentos.")

# =====================
# FUNCIÓN DE CONVERSIÓN ESPECÍFICA
# =====================
def convert_prima(series: pd.Series) -> pd.Series:
    clave = np.array(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ{0123456789"))
    valor = np.array(list("1234567891234567890000000000123456789"))
    diccionario_reemplazo = dict(zip(clave, valor))
    series = series.fillna("0").astype(str)
    return pd.Series(
        np.where(
            (series == "0") | (series == ""),
            series,
            series.str[:-1] + series.str[-1].map(diccionario_reemplazo).fillna("")
        ),
        index=series.index
    )

# =====================
# PROCESAMIENTO DE TXT PARA PRÉSTAMOS (SIN GUARDAR PARQUET GENERAL)
# =====================
def process_txt_file_prestamos(txt_file, file_name: str) -> pd.DataFrame:
    """
    Procesa un archivo TXT a DataFrame (sin guardar el parquet general) y filtra
    el DataFrame para obtener únicamente los registros de préstamos.
    """
    fecha_recepcion = extract_fecha_recepcion(file_name)
    if not fecha_recepcion:
        log(f"❌ No se pudo extraer la fecha de recepción del archivo: {file_name}")
        return pd.DataFrame()

    file_content = txt_file.read()

    if b'\r\n' in file_content:
      lines = file_content.decode('latin-1').splitlines()
      log("Archivo tipo DOS")
    else:
      lines = file_content.decode('utf-8', errors='replace').splitlines()
      log("Archivo tipo UNIX")

    cumulative_indices = [0] + list(accumulate(col_widths))
    data = []

    for line in lines:

        row = {col: line[start:end].strip()
               for col, start, end in zip(column_names, cumulative_indices[:-1], cumulative_indices[1:])}
        row.update({
            "trama_original": line.rstrip("\n"),
            "nombre_archivo_trama": file_name,
            "fecha_recepcion": fecha_recepcion
        })
        data.append(row)
    df = pd.DataFrame(data, columns=column_names + ["trama_original", "nombre_archivo_trama", "fecha_recepcion"])
    df["fecha_recepcion"] = pd.to_datetime(df["fecha_recepcion"], format="%Y-%m-%d").dt.date
    columnas_primas = ["Monto_asegurado", "Prima", "Sumatoria_Capital_Cumulo"]
    for col in columnas_primas:
        if col in df.columns:
            df[col] = pd.to_numeric(convert_prima(df[col]), errors="coerce") / 100.0

    # Filtrar solo los registros de préstamos según 'Tipo de seguro'
    prestamos_values = ['901', '902', '903', '906', '941', '943', '972', '959',
                        '961', '968', '963', '964', '962', '951', '958', '953', '954', '952', '966']
    df_prestamos = df[df["Tipo_de_seguro"].isin(prestamos_values)]
    return df_prestamos

# =====================
# CARGA PARQUET A BQ (Control de duplicados) - SOLO PARQUETS FILTRADOS
# =====================
def load_parquet_to_bigquery():
    """
    Carga los archivos Parquet filtrados (que terminan en '_prestamos.parquet') de GCS
    a la tabla principal en BigQuery. Una vez que la carga es exitosa, se registra
    el archivo en la tabla de control para evitar duplicados.
    """
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_PRIMA_ID}"
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition="WRITE_APPEND"
    )
    bucket = storage_client.bucket(BUCKET_NAME)
    prefix_parquet = GCS_TEMP_PATH.replace(f"gs://{BUCKET_NAME}/", "")
    blobs = list(bucket.list_blobs(prefix=prefix_parquet))
    if not blobs:
        log("⚠️ No se encontraron archivos PARQUET en la ruta.")
        return
    for blob in blobs:
        if blob.name.endswith("_prestamos.parquet"):
            parquet_name = os.path.basename(blob.name)
            if check_if_parquet_processed(parquet_name):
                log(f"🔄 Parquet ya cargado: {parquet_name}. Omitiendo...")
                continue
            parquet_uri = f"gs://{BUCKET_NAME}/{blob.name}"
            log(f"🚀 Cargando {parquet_name} a BigQuery...")
            load_job = bigquery_client.load_table_from_uri(
                parquet_uri,
                table_id,
                job_config=job_config
            )
            load_job.result()
            log(f"✅ Cargado Parquet {parquet_name} en BigQuery.")
            # Registrar en control SÓLO DESPUÉS de cargar a BigQuery.
            insert_control_record(parquet_name, "CARGADO_PARQUET")
    log("🚀 Carga de todos los Parquets filtrados completada en BigQuery.")

# =====================
# PROCESAR ZIP
# =====================
def process_zip_files():
    """
    Procesa archivos ZIP en GCS, extrayendo los archivos TXT que matchean PATTERN
    (incluso en subcarpetas) y evita reprocesarlos si ya existen los parquets.
    Para cada TXT, se filtra el DataFrame para obtener solo los registros de préstamos,
    y se guarda este resultado filtrado en un parquet (con sufijo '_prestamos.parquet').
    Luego se registra el ZIP como procesado.
    """
    bucket = storage_client.bucket(BUCKET_NAME)
    blobs = list(bucket.list_blobs(prefix=PREFIX))
    if not blobs:
        log("⚠️ No se encontraron archivos en el bucket.")
        return
    for blob in blobs:
        if blob.name.endswith(".zip"):
            file_name = os.path.basename(blob.name)
            if check_if_zip_processed(file_name):
                log(f"🔄 Archivo ZIP ya procesado: {file_name}. Omitiendo...")
                continue
            log(f"\n📦 Procesando ZIP: {file_name}")
            zip_bytes = blob.download_as_bytes()
            with zipfile.ZipFile(io.BytesIO(zip_bytes), "r") as z:
                all_files = z.namelist()
                log(f"Archivos dentro del ZIP: {all_files}")
                archivos_txt = []
                for path_in_zip in all_files:
                    base = os.path.basename(path_in_zip)
                    if PATTERN.match(base):
                        archivos_txt.append(path_in_zip)
                if not archivos_txt:
                    log(f"⚠️ No se encontraron TXT válidos en {file_name}. Omitiendo...")
                    continue
                for txt_path in archivos_txt:
                    base_txt_name = os.path.basename(txt_path)
                    # Revisar si ya existe el parquet filtrado
                    parquet_file_filtered = f"{base_txt_name}_prestamos.parquet"
                    full_parquet_gcs_path_filtered = f"{GCS_TEMP_PATH}{parquet_file_filtered}"
                    parquet_blob_filtered = bucket.blob(full_parquet_gcs_path_filtered.replace(f"gs://{BUCKET_NAME}/", ""))
                    if parquet_blob_filtered.exists():
                        log(f"🔄 Parquet filtrado {parquet_file_filtered} ya existe. Omitiendo procesamiento de {base_txt_name}...")
                        continue
                    log(f"→ Procesando archivo: {txt_path} (basename={base_txt_name})")
                    with z.open(txt_path) as txt_file:
                        file_bytes = txt_file.read()
                        # Procesamiento filtrado para préstamos
                        df_prestamos = process_txt_file_prestamos(io.BytesIO(file_bytes), base_txt_name)
                        if not df_prestamos.empty:
                            # Convertir el DataFrame a tabla PyArrow sin preservar el índice.
                            table = pa.Table.from_pandas(df_prestamos, preserve_index=False)
                            pq.write_table(table, full_parquet_gcs_path_filtered)
                            log(f"📤 Guardado {full_parquet_gcs_path_filtered} en GCS.")
                        else:
                            log(f"⚠️ No se encontraron registros de préstamos en {base_txt_name}.")
            # Una vez finalizado el procesamiento de todos los TXT del ZIP, se registra el ZIP.
            insert_control_record(file_name, "CARGADO")

# =====================
# MAIN
# =====================
if __name__ == "__main__":
    log("🚀 Iniciando proceso...")
    create_table_if_not_exists()
    create_control_table()
    process_zip_files()
    load_parquet_to_bigquery()
    log("✅ Proceso finalizado.")


[2025-08-05 16:52:55] 🚀 Iniciando proceso...
[2025-08-05 16:52:55] >>> Esquema a usar para crear la tabla:
[2025-08-05 16:52:55] ✅ La tabla TABLA_PRESTAMOS_BBVA ya existe (no se modifica).
[2025-08-05 16:52:55] ✅ La tabla de control TABLA_CONTROL_PREST ya existe.
[2025-08-05 16:52:58] 🔄 Archivo ZIP ya procesado: 04_Abril25.zip. Omitiendo...
[2025-08-05 16:53:00] 🔄 Archivo ZIP ya procesado: 05_Mayo25.zip. Omitiendo...
[2025-08-05 16:53:02] 🔄 Archivo ZIP ya procesado: 06_Junio25.zip. Omitiendo...
[2025-08-05 16:53:04] 
📦 Procesando ZIP: 07_Julio25.zip
[2025-08-05 16:53:05] Archivos dentro del ZIP: ['07_Julio25/RED010725_0207.TXT', '07_Julio25/RED020725_0307.TXT', '07_Julio25/RED030725_0407.TXT', '07_Julio25/RED040725_0707.TXT', '07_Julio25/RED070725_0807.TXT', '07_Julio25/RED080725_0907.TXT', '07_Julio25/RED090725_1007.TXT', '07_Julio25/RED100725_1107.TXT', '07_Julio25/RED110725_1407.TXT', '07_Julio25/RED140725_1507.TXT', '07_Julio25/RED150725_1607.TXT', '07_Julio25/RED160725_1707.TXT', 